In [1]:
import pandas as pd
import os
import re
import numpy as np
from difflib import SequenceMatcher

dataset_path = "/kaggle/input/datasets/harshkaushik99/ml-dataset/student_resource/dataset"

print(os.listdir(dataset_path))

train_path = "/kaggle/input/datasets/harshkaushik99/ml-dataset/student_resource/dataset/train"
test_path = "/kaggle/input/datasets/harshkaushik99/ml-dataset/student_resource/dataset/test"

print("TRAIN FILES:")
print(os.listdir(train_path))

print("\nTEST FILES:")
print(os.listdir(test_path))

base = "/kaggle/input/datasets/harshkaushik99/ml-dataset/student_resource/dataset"

train_path = os.path.join(base, "train")
test_path = os.path.join(base, "test")

files = [
    "train_ground_truth.tsv",
    "train_source1.tsv",
    "train_source2.tsv",
    "train_source3.tsv"
]

for file in files:
    path = os.path.join(train_path, file)
    df = pd.read_csv(path, sep="\t")
    print("\n" + "=" * 70)
    print(file)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nFirst 5 rows:")
    display(df.head())

test_files = [
    "test_source1.tsv",
    "test_source2.tsv",
    "test_source3.tsv"
]

for file in test_files:
    path = os.path.join(test_path, file)
    df = pd.read_csv(path, sep="\t")
    print("\n" + "=" * 70)
    print(file)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nFirst 5 rows:")
    display(df.head())

s1 = pd.read_csv(os.path.join(train_path, "train_source1.tsv"), sep="\t")
s2 = pd.read_csv(os.path.join(train_path, "train_source2.tsv"), sep="\t")
s3 = pd.read_csv(os.path.join(train_path, "train_source3.tsv"), sep="\t")

print("===== S2 MISSING VALUES =====")
print(s2.isna().sum())

print("\n===== S3 MISSING VALUES =====")
print(s3.isna().sum())

print("===== S2 COUNTRY DISTRIBUTION =====")
print(s2["country"].value_counts())

print("\n===== S3 COUNTRY DISTRIBUTION =====")
print(s3["country"].value_counts())

gt = pd.read_csv(os.path.join(train_path, "train_ground_truth.tsv"), sep="\t")

print("Ground Truth shape:", gt.shape)
print(gt.head())
print(gt.head(10))

print("\n===== COLUMNS =====")
print(gt.columns)

print("\n===== TOTAL ROWS =====")
print(len(gt))

print("\n===== EMPTY MATCHES =====")
print((gt["matched_entity_ids"].fillna("").str.strip() == "").sum())

print("\n===== NON-EMPTY MATCHES =====")
print((gt["matched_entity_ids"].fillna("").str.strip() != "").sum())

match_count = (
    gt["matched_entity_ids"]
    .fillna("")
    .apply(lambda x: 0 if x.strip() == "" else len(x.split(",")))
)

print("===== MATCH COUNT PER S1 =====")
print(match_count.describe())

print("\n===== MATCH COUNT DISTRIBUTION =====")
print(match_count.value_counts().sort_index().head(20))

print("\n===== MAX MATCHES FOR ONE S1 =====")
print(match_count.max())

print("===== SAMPLE S1 =====")
print(s1[["business_name", "business_address", "country"]].head(10).to_string(index=False))

print("\n===== SAMPLE S2 =====")
print(s2[["business_name", "business_address", "country"]].head(10).to_string(index=False))

print("\n===== SAMPLE S3 =====")
print(s3[["business_name", "business_address", "country"]].head(10).to_string(index=False))


def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\u0900-\u097f]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


print("===== NAME NORMALIZATION =====")
for x in s1["business_name"].head(10):
    print("RAW :", x)
    print("NORM:", normalize_text(x))
    print()

print("===== ADDRESS NORMALIZATION =====")
for x in s1["business_address"].head(10):
    print("RAW :", x)
    print("NORM:", normalize_text(x))
    print()


def normalize_name(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    replacements = {
        "private limited": "pvt ltd",
        "private ltd": "pvt ltd",
        "incorporated": "inc",
        "limited": "ltd"
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    text = re.sub(r"[^a-z0-9\u0900-\u097f]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


print("===== NAME NORMALIZATION V2 =====")
for x in s1["business_name"].head(10):
    print("RAW :", x)
    print("NORM:", normalize_name(x))
    print()

s2_sample = s2.head(10000).copy()
s3_sample = s3.head(10000).copy()

s2_sample["name_norm"] = s2_sample["business_name"].apply(normalize_name)
s3_sample["name_norm"] = s3_sample["business_name"].apply(normalize_name)

print("S2 unique normalized names:", s2_sample["name_norm"].nunique())
print("S3 unique normalized names:", s3_sample["name_norm"].nunique())
print("\nS2 duplicate normalized names:", len(s2_sample) - s2_sample["name_norm"].nunique())
print("S3 duplicate normalized names:", len(s3_sample) - s3_sample["name_norm"].nunique())

sample_gt = gt[gt["matched_entity_ids"].fillna("").str.strip() != ""].head(1000).copy()

print("Sample S1 records:", len(sample_gt))
print(sample_gt.head())

pairs = sample_gt[["source1_entity_id", "matched_entity_ids"]].copy()
pairs["matched_entity_ids"] = pairs["matched_entity_ids"].str.split(",")
pairs = pairs.explode("matched_entity_ids")
pairs = pairs.rename(columns={"matched_entity_ids": "matched_id"})

print("Total true pairs:", len(pairs))
print(pairs.head(10))

s1_lookup = s1[["entity_id", "business_name", "country"]].copy()
s1_lookup["name_norm"] = s1_lookup["business_name"].apply(normalize_name)
s1_lookup = s1_lookup.rename(columns={
    "entity_id": "source1_entity_id",
    "business_name": "s1_name",
    "name_norm": "s1_name_norm"
})

s2_lookup = s2[["entity_id", "business_name", "country"]].copy()
s3_lookup = s3[["entity_id", "business_name", "country"]].copy()
s2_lookup["source"] = "S2"
s3_lookup["source"] = "S3"

candidates = pd.concat([s2_lookup, s3_lookup], ignore_index=True)
candidates["name_norm"] = candidates["business_name"].apply(normalize_name)
candidates = candidates.rename(columns={
    "entity_id": "matched_id",
    "business_name": "matched_name"
})

pairs_check = pairs.merge(s1_lookup, on="source1_entity_id", how="left")
pairs_check = pairs_check.merge(
    candidates[["matched_id", "matched_name", "name_norm", "country"]],
    on="matched_id",
    how="left",
    suffixes=("_s1", "_matched")
)

pairs_check["exact_name_match"] = (pairs_check["s1_name_norm"] == pairs_check["name_norm"])

print("Total true pairs:", len(pairs_check))
print("\nExact normalized-name matches:")
print(pairs_check["exact_name_match"].value_counts())
print("\nPercentage:")
print(pairs_check["exact_name_match"].mean() * 100)


def name_similarity(a, b):
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


pairs_check["name_similarity"] = pairs_check.apply(
    lambda row: name_similarity(row["s1_name_norm"], row["name_norm"]),
    axis=1
)

print("===== NAME SIMILARITY =====")
print(pairs_check["name_similarity"].describe())

print("\n===== SIMILARITY BUCKETS =====")
bins = [0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0]
print(
    pd.cut(pairs_check["name_similarity"], bins=bins, include_lowest=True)
    .value_counts().sort_index()
)

# S2 + S3 address lookup
s2_addr = s2[["entity_id", "business_address"]].copy()
s3_addr = s3[["entity_id", "business_address"]].copy()

candidate_addr = pd.concat([s2_addr, s3_addr], ignore_index=True)
candidate_addr = candidate_addr.rename(columns={"entity_id": "matched_id"})

print(candidate_addr.head())
print("\nTotal S2 + S3 records:", len(candidate_addr))

s1_addr = s1[["entity_id", "business_address"]].copy()
s1_addr = s1_addr.rename(columns={
    "entity_id": "source1_entity_id",
    "business_address": "s1_address"
})

pairs_check = pairs_check.drop(columns=["s1_address", "matched_address"], errors="ignore")
pairs_check = pairs_check.merge(s1_addr, on="source1_entity_id", how="left")
pairs_check = pairs_check.merge(candidate_addr, on="matched_id", how="left")

print(pairs_check[["source1_entity_id", "matched_id", "s1_address", "business_address"]].head())

pairs_check["s1_address_norm"] = pairs_check["s1_address"].apply(normalize_text)
pairs_check["matched_address_norm"] = pairs_check["business_address"].apply(normalize_text)

pairs_check["address_similarity"] = pairs_check.apply(
    lambda row: name_similarity(row["s1_address_norm"], row["matched_address_norm"]),
    axis=1
)

print("===== ADDRESS SIMILARITY =====")
print(pairs_check["address_similarity"].describe())

print("\n===== ADDRESS SIMILARITY BUCKETS =====")
print(
    pd.cut(pairs_check["address_similarity"], bins=bins, include_lowest=True)
    .value_counts().sort_index()
)

pairs_check["combined_score"] = (
    0.6 * pairs_check["name_similarity"] + 0.4 * pairs_check["address_similarity"]
)

print(pairs_check[[
    "source1_entity_id", "matched_id", "name_similarity",
    "address_similarity", "combined_score"
]].head(20))

print(pairs_check["combined_score"].describe())
print(
    pd.cut(pairs_check["combined_score"], bins=[0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0])
    .value_counts().sort_index()
)

np.random.seed(42)

s1_ids = sample_gt["source1_entity_id"].values
candidate_ids = candidates["matched_id"].values

random_ids = np.random.choice(candidate_ids, size=len(sample_gt), replace=False)

false_base = pd.DataFrame({
    "source1_entity_id": s1_ids,
    "matched_id": random_ids
})

print(false_base.head())

# --- Build false_check BEFORE using it (this was the ordering bug) ---
candidate_names = candidates[["matched_id", "matched_name"]].drop_duplicates("matched_id")

false_check = false_base.merge(
    s1[["entity_id", "business_name", "business_address"]],
    left_on="source1_entity_id",
    right_on="entity_id",
    how="left"
).drop(columns=["entity_id"])

false_check = false_check.merge(candidate_names, on="matched_id", how="left")

candidate_addr_clean = candidate_addr[["matched_id", "business_address"]].drop_duplicates("matched_id")

false_check = false_check.merge(
    candidate_addr_clean,
    on="matched_id",
    how="left",
    suffixes=("_s1", "_candidate")
)

print(false_check.columns.tolist())

false_check["s1_name_norm"] = false_check["business_name"].apply(normalize_name)
false_check["matched_name_norm"] = false_check["matched_name"].apply(normalize_name)

false_check["name_similarity"] = false_check.apply(
    lambda row: name_similarity(row["s1_name_norm"], row["matched_name_norm"]),
    axis=1
)

false_check["address_similarity"] = false_check.apply(
    lambda row: name_similarity(
        normalize_text(row["business_address_s1"]),
        normalize_text(row["business_address_candidate"])
    ),
    axis=1
)

false_check["combined_score"] = (
    0.6 * false_check["name_similarity"] + 0.4 * false_check["address_similarity"]
)

print(false_check[[
    "source1_entity_id", "matched_id", "name_similarity",
    "address_similarity", "combined_score"
]].head(10))

print("===== TRUE MATCHES =====")
print(pairs_check["combined_score"].describe())

print("\n===== RANDOM FALSE MATCHES =====")
print(false_check["combined_score"].describe())

print("===== TRUE MATCH BUCKETS =====")
print(
    pd.cut(pairs_check["combined_score"], bins=bins, include_lowest=True)
    .value_counts().sort_index()
)

print("\n===== FALSE MATCH BUCKETS =====")
print(
    pd.cut(false_check["combined_score"], bins=bins, include_lowest=True)
    .value_counts().sort_index()
)

for t in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    kept = (pairs_check["combined_score"] >= t).sum()
    total = len(pairs_check)
    print(f"Threshold {t}: {kept}/{total} true matches kept ({kept/total:.2%})")

s1_sample = s1[s1["entity_id"].isin(sample_gt["source1_entity_id"])][
    ["entity_id", "business_name", "business_address"]
].copy()

s1_sample["name_norm"] = s1_sample["business_name"].apply(normalize_name)

target_names = set(s1_sample["name_norm"])

hard_pool = candidates[candidates["name_norm"].isin(target_names)][
    ["matched_id", "matched_name", "country", "source", "name_norm"]
].drop_duplicates("matched_id")

print("Same-name candidate records:", len(hard_pool))
print(hard_pool.head())

true_pairs = set()
for _, row in sample_gt.iterrows():
    ids = str(row["matched_entity_ids"]).strip()
    if ids:
        for mid in ids.split(","):
            true_pairs.add((row["source1_entity_id"], mid.strip()))

hard_negatives = s1_sample.merge(hard_pool, on="name_norm", how="inner")

hard_negatives = hard_negatives[
    ~hard_negatives.apply(
        lambda row: (row["entity_id"], row["matched_id"]) in true_pairs,
        axis=1
    )
].copy()

hard_negatives = hard_negatives.rename(columns={
    "entity_id": "source1_entity_id",
    "business_address": "s1_address",
    "matched_name": "candidate_name"
})

print("Hard negative pairs:", len(hard_negatives))
print(hard_negatives[[
    "source1_entity_id", "matched_id", "business_name", "candidate_name"
]].head(20))

hard_addr = candidate_addr[["matched_id", "business_address"]].drop_duplicates("matched_id")

hard_negatives = hard_negatives.merge(hard_addr, on="matched_id", how="left")

hard_negatives["address_similarity"] = hard_negatives.apply(
    lambda row: name_similarity(
        normalize_text(row["s1_address"]),
        normalize_text(row["business_address"])
    ),
    axis=1
)

hard_negatives["name_similarity"] = hard_negatives.apply(
    lambda row: name_similarity(
        normalize_name(row["business_name"]),
        normalize_name(row["candidate_name"])
    ),
    axis=1
)

hard_negatives["combined_score"] = (
    0.6 * hard_negatives["name_similarity"] + 0.4 * hard_negatives["address_similarity"]
)

print(hard_negatives[[
    "source1_entity_id", "matched_id", "name_similarity",
    "address_similarity", "combined_score"
]].head(20))

print("\n===== HARD NEGATIVE SUMMARY =====")
print(hard_negatives["combined_score"].describe())

print("\n===== HARD NEGATIVE BUCKETS =====")
print(
    pd.cut(hard_negatives["combined_score"], bins=[0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0], include_lowest=True)
    .value_counts().sort_index()
)

# Exact-name TRUE matches
true_exact_name = pairs_check[pairs_check["name_similarity"] == 1.0].copy()

print("Exact-name TRUE pairs:", len(true_exact_name))
print("\n===== TRUE EXACT-NAME ADDRESS =====")
print(true_exact_name["address_similarity"].describe())

print("\n===== HARD-NEGATIVE ADDRESS =====")
print(hard_negatives["address_similarity"].describe())

bins2 = [0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0]

print("===== TRUE EXACT-NAME =====")
print(
    pd.cut(true_exact_name["address_similarity"], bins=bins2, include_lowest=True)
    .value_counts().sort_index()
)

print("\n===== HARD NEGATIVES =====")
print(
    pd.cut(hard_negatives["address_similarity"], bins=bins2, include_lowest=True)
    .value_counts().sort_index()
)


def address_tokens(text):
    if pd.isna(text):
        return set()
    text = normalize_text(text)
    return set(text.split())


def address_numbers(text):
    if pd.isna(text):
        return set()
    text = str(text).lower()
    nums = re.findall(r'\d+', text)
    return set(nums)


def token_jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def number_overlap(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


true_exact_name["s1_addr_tokens"] = true_exact_name["s1_address"].apply(address_tokens)
true_exact_name["candidate_addr_tokens"] = true_exact_name["business_address"].apply(address_tokens)

true_exact_name["address_token_jaccard"] = true_exact_name.apply(
    lambda row: token_jaccard(row["s1_addr_tokens"], row["candidate_addr_tokens"]),
    axis=1
)

true_exact_name["s1_addr_numbers"] = true_exact_name["s1_address"].apply(address_numbers)
true_exact_name["candidate_addr_numbers"] = true_exact_name["business_address"].apply(address_numbers)

true_exact_name["number_overlap"] = true_exact_name.apply(
    lambda row: number_overlap(row["s1_addr_numbers"], row["candidate_addr_numbers"]),
    axis=1
)

print(true_exact_name[["address_similarity", "address_token_jaccard", "number_overlap"]].describe())

hard_negatives["s1_addr_tokens"] = hard_negatives["s1_address"].apply(address_tokens)
hard_negatives["candidate_addr_tokens"] = hard_negatives["business_address"].apply(address_tokens)

hard_negatives["address_token_jaccard"] = hard_negatives.apply(
    lambda row: token_jaccard(row["s1_addr_tokens"], row["candidate_addr_tokens"]),
    axis=1
)

hard_negatives["s1_addr_numbers"] = hard_negatives["s1_address"].apply(address_numbers)
hard_negatives["candidate_addr_numbers"] = hard_negatives["business_address"].apply(address_numbers)

hard_negatives["number_overlap"] = hard_negatives.apply(
    lambda row: number_overlap(row["s1_addr_numbers"], row["candidate_addr_numbers"]),
    axis=1
)

print(hard_negatives[["address_similarity", "address_token_jaccard", "number_overlap"]].describe())

true_exact_name["new_score"] = (
    0.40 * true_exact_name["name_similarity"] +
    0.20 * true_exact_name["address_similarity"] +
    0.20 * true_exact_name["address_token_jaccard"] +
    0.20 * true_exact_name["number_overlap"]
)

hard_negatives["new_score"] = (
    0.40 * hard_negatives["name_similarity"] +
    0.20 * hard_negatives["address_similarity"] +
    0.20 * hard_negatives["address_token_jaccard"] +
    0.20 * hard_negatives["number_overlap"]
)

print("TRUE MATCHES")
print(true_exact_name["new_score"].describe())

print("\nWRONG SAME-NAME MATCHES")
print(hard_negatives["new_score"].describe())

thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]

for t in thresholds:
    true_keep = (true_exact_name["new_score"] >= t).sum()
    wrong_keep = (hard_negatives["new_score"] >= t).sum()

    true_pct = true_keep / len(true_exact_name) * 100
    wrong_pct = wrong_keep / len(hard_negatives) * 100

    print(
        f"Threshold {t:.2f} | "
        f"True kept: {true_keep}/{len(true_exact_name)} ({true_pct:.1f}%) | "
        f"Wrong kept: {wrong_keep}/{len(hard_negatives)} ({wrong_pct:.2f}%)"
    )


def calculate_f05(true_ids, pred_ids):
    true_set = set(true_ids)
    pred_set = set(pred_ids)

    if len(true_set) == 0 and len(pred_set) == 0:
        return 1.0
    if len(true_set) == 0 or len(pred_set) == 0:
        return 0.0

    tp = len(true_set & pred_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0

    beta = 0.5
    if precision == 0 and recall == 0:
        return 0.0

    return (1 + beta ** 2) * precision * recall / (beta ** 2 * precision + recall)


print(calculate_f05(["A", "B", "C"], ["A", "B", "C"]))

print(pairs_check.columns.tolist())

pairs_check["s1_addr_tokens"] = pairs_check["s1_address"].apply(address_tokens)
pairs_check["candidate_addr_tokens"] = pairs_check["business_address"].apply(address_tokens)

pairs_check["address_token_jaccard"] = pairs_check.apply(
    lambda row: token_jaccard(row["s1_addr_tokens"], row["candidate_addr_tokens"]),
    axis=1
)

pairs_check["s1_addr_numbers"] = pairs_check["s1_address"].apply(address_numbers)
pairs_check["candidate_addr_numbers"] = pairs_check["business_address"].apply(address_numbers)

pairs_check["number_overlap"] = pairs_check.apply(
    lambda row: number_overlap(row["s1_addr_numbers"], row["candidate_addr_numbers"]),
    axis=1
)

pairs_check["new_score"] = (
    0.40 * pairs_check["name_similarity"]
    + 0.20 * pairs_check["address_similarity"]
    + 0.20 * pairs_check["address_token_jaccard"]
    + 0.20 * pairs_check["number_overlap"]
)

print(pairs_check[[
    "name_similarity", "address_similarity", "address_token_jaccard",
    "number_overlap", "new_score"
]].head())

print("sample_gt:", sample_gt.shape)
print("pairs_check:", pairs_check.shape)
print("\nUnique S1 in sample_gt:", sample_gt["source1_entity_id"].nunique())
print("Unique S1 in pairs_check:", pairs_check["source1_entity_id"].nunique())

thresholds = [0.55, 0.60, 0.65, 0.70]
results = []

gt_dict = dict(zip(sample_gt["source1_entity_id"], sample_gt["matched_entity_ids"]))

for threshold in thresholds:
    scores = []
    for s1_id, group in pairs_check.groupby("source1_entity_id"):
        true_ids = str(gt_dict[s1_id]).split(",")
        pred_ids = group.loc[group["new_score"] >= threshold, "matched_id"].tolist()
        score = calculate_f05(true_ids, pred_ids)
        scores.append(score)

    mean_f05 = sum(scores) / len(scores)
    results.append({"threshold": threshold, "mean_f05": mean_f05})

results_df = pd.DataFrame(results)
print(results_df)

total_true_pairs = len(pairs_check)
exact_true_pairs = pairs_check["exact_name_match"].sum()
recall = exact_true_pairs / total_true_pairs

print("Total true pairs:", total_true_pairs)
print("True pairs found by exact normalized name:", exact_true_pairs)
print("Blocking recall:", round(recall * 100, 2), "%")

get_ipython().system('pip -q install rapidfuzz')
from rapidfuzz import process, fuzz  # noqa: F401

candidate_names = candidates[["name_norm"]].drop_duplicates()
candidate_name_list = candidate_names["name_norm"].tolist()

print("Unique candidate names:", len(candidate_name_list))


def name_block_key(text):
    if not text:
        return ("", 0)
    first_char = text[0]
    length_bucket = len(text) // 5
    return (first_char, length_bucket)


candidate_names["block"] = candidate_names["name_norm"].apply(name_block_key)
block_counts = candidate_names["block"].value_counts()

print("Total blocks:", len(block_counts))
print("\nLargest blocks:")
print(block_counts.head(10))


def better_name_block(text):
    if not text:
        return ("", 0)
    text = text.strip()
    prefix = text[:2]
    length_bucket = len(text) // 5
    return (prefix, length_bucket)


candidate_names["better_block"] = candidate_names["name_norm"].apply(better_name_block)
better_counts = candidate_names["better_block"].value_counts()

print("Total blocks:", len(better_counts))
print("\nLargest blocks:")
print(better_counts.head(10))

candidate_names = candidates[["name_norm", "country"]].drop_duplicates()

candidate_names["final_block"] = (
    candidate_names["country"].astype(str)
    + "_"
    + candidate_names["name_norm"].str[:2]
    + "_"
    + (candidate_names["name_norm"].str.len() // 5).astype(str)
)

final_counts = candidate_names["final_block"].value_counts()

print("Total blocks:", len(final_counts))
print("\nLargest blocks:")
print(final_counts.head(10))

candidate_names["ngram_block"] = (
    candidate_names["country"].astype(str) + "_" + candidate_names["name_norm"].str[:3]
)

ngram_counts = candidate_names["ngram_block"].value_counts()

print("Total blocks:", len(ngram_counts))
print("\nLargest blocks:")
print(ngram_counts.head(10))

print("\nMedian block size:", ngram_counts.median())
print("Mean block size:", round(ngram_counts.mean(), 2))

s2_candidates = s2[["entity_id", "business_name", "country"]].copy()
s2_candidates["source"] = "source2"

s3_candidates = s3[["entity_id", "business_name", "country"]].copy()
s3_candidates["source"] = "source3"

candidates = pd.concat([s2_candidates, s3_candidates], ignore_index=True)

candidates["name_norm"] = candidates["business_name"].apply(normalize_name)

candidates = candidates.rename(columns={
    "entity_id": "matched_id",
    "business_name": "matched_name"
})

print(candidates.shape)
print(candidates.columns.tolist())

['test', 'train']
TRAIN FILES:
['train_ground_truth.tsv', 'train_source3.tsv', 'train_source2.tsv', 'train_source1.tsv']

TEST FILES:
['test_source2.tsv', 'test_source3.tsv', 'test_source1.tsv']

train_ground_truth.tsv
Shape: (2206821, 2)
Columns: ['source1_entity_id', 'matched_entity_ids']

First 5 rows:


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."



train_source1.tsv
Shape: (2206821, 4)
Columns: ['entity_id', 'business_name', 'business_address', 'country']

First 5 rows:


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India



train_source2.tsv
Shape: (5034616, 4)
Columns: ['entity_id', 'business_name', 'business_address', 'country']

First 5 rows:


...


test_source2.tsv
Shape: (4887273, 4)
Columns: ['entity_id', 'business_name', 'business_address', 'country']

First 5 rows:


,entity_id,business_name,business_address,country
0,S2-192345572,Brahma Infosoft,"COIMATORE COLONY, HUNSUR TQMYSORE DIST., Karna...",India
1,S2-566025912,Marina Ecole France Sarl,"63 R. DE DIEPPE, LILLE, Hauts-de-France",France
2,S2-158121477,SCI Ptit Àmicale,"18 RUE JEN ZAY, Dunkerque, Nord",France
3,S2-89663826,Apex Summit,"67 KENTUCKY ST, SALYERSVILLE, KY",US
4,S2-884102769,Fresh Truist,"8264 FILLY COURT, ROANOKE COUNTY, VA",US



test_source3.tsv
Shape: (5082316, 4)
Columns: ['entity_id', 'business_name', 'business_address', 'country']

First 5 rows:


,entity_id,business_name,business_address,country
0,S3-462677478,मॉडर्न फाइनेंस,"No 10 Enkay Square, 448A, Udyog Vihar Phase V,...",India
1,S3-374810425,Shri Sai Infratech Co,"3/115, East Delhi, DL",India
2,S3-198586129,Fractales Amis Groupe S.A.S,"23 Rue Icmre, La Teste-de-buch, Gironde",France
3,S3-10300249,Shri Supreme Consulting Private (Limited),"H.no 910 A 3503, Mumbai, महाराष्ट्र",India
4,S3-604980231,Prime Realty Ventures Public Limited,"G.t. Karnal Road, Industrial Area, New Delhi, ...",India


===== S2 MISSING VALUES =====
entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64

===== S3 MISSING VALUES =====
entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64
===== S2 COUNTRY DISTRIBUTION =====
country
US       3016817
India    2017799
Name: count, dtype: int64

===== S3 COUNTRY DISTRIBUTION =====
country
US       3170056
India    2115547
Name: count, dtype: int64
Ground Truth shape: (2206821, 2)
  source1_entity_id                                 matched_entity_ids
0         S1-965667  S2-681193310,S2-743505751,S3-775321672,S3-1129...
1       S1-55344266  S2-249013014,S2-197070651,S3-478195123,S3-3843...
2      S1-343815751             S2-790675320,S2-479876582,S3-878454467
3      S1-656753428              S2-153058913,S2-24659151,S3-679606215
4      S1-102811957  S2-478959098,S2-553508714,S2-625774905,S3-7280...
  source1_entity_id          

In [2]:
# S1 lookup
s1_lookup = s1.set_index("entity_id")

# Sirf non-empty ground truth mein se first 1000 validation rows
sample_gt = gt[
    gt["matched_entity_ids"].fillna("").str.strip() != ""
].head(1000).copy()

# S1 information add karo
sample_gt["s1_name_norm"] = sample_gt["source1_entity_id"].map(
    s1_lookup["business_name"].apply(normalize_name)
)

sample_gt["country_s1"] = sample_gt["source1_entity_id"].map(
    s1_lookup["country"]
)

# 3-character + country block
sample_gt["ngram_block"] = (
    sample_gt["country_s1"].astype(str)
    + "_"
    + sample_gt["s1_name_norm"].fillna("").str[:3]
)

candidates["ngram_block"] = (
    candidates["country"].astype(str)
    + "_"
    + candidates["name_norm"].fillna("").str[:3]
)

# Har block ke candidate IDs
block_to_ids = (
    candidates.groupby("ngram_block")["matched_id"]
    .apply(set)
    .to_dict()
)

# Recall calculate
found_pairs = 0
total_pairs = 0

for _, row in sample_gt.iterrows():

    true_ids = [
        x.strip()
        for x in str(row["matched_entity_ids"]).split(",")
        if x.strip()
    ]

    block_ids = block_to_ids.get(row["ngram_block"], set())

    total_pairs += len(true_ids)
    found_pairs += sum(x in block_ids for x in true_ids)

recall = found_pairs / total_pairs

print("Total true pairs:", total_pairs)
print("True pairs found:", found_pairs)
print("Blocking recall:", round(recall * 100, 2), "%")

Total true pairs: 3589
True pairs found: 2728
Blocking recall: 76.01 %


In [3]:
# Exact name + country block
candidates["exact_block"] = (
    candidates["country"].astype(str)
    + "_"
    + candidates["name_norm"].fillna("")
)

sample_gt["exact_block"] = (
    sample_gt["country_s1"].astype(str)
    + "_"
    + sample_gt["s1_name_norm"].fillna("")
)

# Dictionaries
ngram_to_ids = (
    candidates.groupby("ngram_block")["matched_id"]
    .apply(set)
    .to_dict()
)

exact_to_ids = (
    candidates.groupby("exact_block")["matched_id"]
    .apply(set)
    .to_dict()
)

# Combined recall
found_pairs = 0
total_pairs = 0

for _, row in sample_gt.iterrows():

    true_ids = [
        x.strip()
        for x in str(row["matched_entity_ids"]).split(",")
        if x.strip()
    ]

    ids_3gram = ngram_to_ids.get(row["ngram_block"], set())
    ids_exact = exact_to_ids.get(row["exact_block"], set())

    combined_ids = ids_3gram | ids_exact

    total_pairs += len(true_ids)
    found_pairs += sum(x in combined_ids for x in true_ids)

recall = found_pairs / total_pairs

print("Total true pairs:", total_pairs)
print("True pairs found:", found_pairs)
print("Combined blocking recall:", round(recall * 100, 2), "%")

Total true pairs: 3589
True pairs found: 2728
Combined blocking recall: 76.01 %


In [4]:
import rapidfuzz

print("RapidFuzz version:", rapidfuzz.__version__)

RapidFuzz version: 3.14.6


In [6]:
# Block + name -> candidate IDs ka fast lookup

block_name_to_ids = (
    candidates
    .groupby(["ngram_block", "name_norm"])["matched_id"]
    .apply(set)
    .to_dict()
)

print("Lookup ready:", len(block_name_to_ids))

Lookup ready: 7441525


In [7]:
from rapidfuzz import process, fuzz

found_pairs = 0
total_pairs = 0

for _, row in sample_gt.iterrows():

    true_ids = [
        x.strip()
        for x in str(row["matched_entity_ids"]).split(",")
        if x.strip()
    ]

    choices = block_name_choices.get(row["ngram_block"], [])

    if len(choices) == 0:
        continue

    # Top 10 similar names
    results = process.extract(
        row["s1_name_norm"],
        choices,
        scorer=fuzz.ratio,
        limit=10,
        score_cutoff=60
    )

    # Candidate IDs directly lookup se
    candidate_ids = set()

    for name, score, _ in results:
        ids = block_name_to_ids.get(
            (row["ngram_block"], name),
            set()
        )
        candidate_ids.update(ids)

    total_pairs += len(true_ids)
    found_pairs += sum(x in candidate_ids for x in true_ids)

recall = found_pairs / total_pairs

print("Total true pairs:", total_pairs)
print("True pairs found:", found_pairs)
print("Fuzzy blocking recall:", round(recall * 100, 2), "%")

Total true pairs: 3589
True pairs found: 2074
Fuzzy blocking recall: 57.79 %


In [8]:
# Sirf validation S1 ke required blocks
validation_blocks = set(sample_gt["ngram_block"])

validation_candidates = candidates[
    candidates["ngram_block"].isin(validation_blocks)
].copy()

print("Validation candidate rows:", len(validation_candidates))
print("Unique candidate IDs:", validation_candidates["matched_id"].nunique())

Validation candidate rows: 4592043
Unique candidate IDs: 4592043


In [9]:
# Country + first 3 chars + name length bucket

candidates["small_block"] = (
    candidates["country"].astype(str)
    + "_"
    + candidates["name_norm"].fillna("").str[:3]
    + "_"
    + (candidates["name_norm"].fillna("").str.len() // 5).astype(str)
)

sample_gt["small_block"] = (
    sample_gt["country_s1"].astype(str)
    + "_"
    + sample_gt["s1_name_norm"].fillna("").str[:3]
    + "_"
    + (sample_gt["s1_name_norm"].fillna("").str.len() // 5).astype(str)
)

# Block -> candidate IDs
small_block_to_ids = (
    candidates.groupby("small_block")["matched_id"]
    .apply(set)
    .to_dict()
)

found_pairs = 0
total_pairs = 0

for _, row in sample_gt.iterrows():

    true_ids = [
        x.strip()
        for x in str(row["matched_entity_ids"]).split(",")
        if x.strip()
    ]

    candidate_ids = small_block_to_ids.get(
        row["small_block"], set()
    )

    total_pairs += len(true_ids)
    found_pairs += sum(x in candidate_ids for x in true_ids)

recall = found_pairs / total_pairs

print("Total true pairs:", total_pairs)
print("True pairs found:", found_pairs)
print("Small-block recall:", round(recall * 100, 2), "%")

Total true pairs: 3589
True pairs found: 1673
Small-block recall: 46.61 %


In [10]:
print(pairs_check.shape)
print(pairs_check.columns.tolist())

(3589, 23)
['source1_entity_id', 'matched_id', 's1_name', 'country_s1', 's1_name_norm', 'matched_name', 'name_norm', 'country_matched', 'exact_name_match', 'name_similarity', 's1_address', 'business_address', 's1_address_norm', 'matched_address_norm', 'address_similarity', 'combined_score', 's1_addr_tokens', 'candidate_addr_tokens', 'address_token_jaccard', 's1_addr_numbers', 'candidate_addr_numbers', 'number_overlap', 'new_score']


In [11]:
import numpy as np

def calculate_f05(true_ids, pred_ids):
    true_set = set(true_ids)
    pred_set = set(pred_ids)

    if len(true_set) == 0 and len(pred_set) == 0:
        return 1.0

    if len(true_set) == 0 or len(pred_set) == 0:
        return 0.0

    tp = len(true_set & pred_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0

    beta = 0.5

    if precision == 0 and recall == 0:
        return 0.0

    return (
        (1 + beta**2) * precision * recall
        / (beta**2 * precision + recall)
    )


thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]

for threshold in thresholds:

    scores = []

    for s1_id, group in pairs_check.groupby("source1_entity_id"):

        true_ids = sample_gt.loc[
            sample_gt["source1_entity_id"] == s1_id,
            "matched_entity_ids"
        ].iloc[0]

        true_ids = [
            x.strip()
            for x in str(true_ids).split(",")
            if x.strip()
        ]

        pred_ids = group.loc[
            group["new_score"] >= threshold,
            "matched_id"
        ].tolist()

        scores.append(calculate_f05(true_ids, pred_ids))

    print(
        f"Threshold {threshold:.2f} -> "
        f"F0.5 = {np.mean(scores):.4f}"
    )

Threshold 0.50 -> F0.5 = 0.9520
Threshold 0.55 -> F0.5 = 0.9260
Threshold 0.60 -> F0.5 = 0.8877
Threshold 0.65 -> F0.5 = 0.8390
Threshold 0.70 -> F0.5 = 0.7789
Threshold 0.75 -> F0.5 = 0.6986
Threshold 0.80 -> F0.5 = 0.6070


In [12]:
print("True pairs:", len(pairs_check))
print("Hard negatives:", len(hard_negatives))

print("\nTrue score:")
print(pairs_check["new_score"].describe())

print("\nHard-negative score:")
print(hard_negatives["new_score"].describe())

True pairs: 3589
Hard negatives: 9927

True score:
count    3589.000000
mean        0.728390
std         0.172005
min         0.113228
25%         0.623611
50%         0.761003
75%         0.863511
max         1.000000
Name: new_score, dtype: float64

Hard-negative score:
count    9927.000000
mean        0.456799
std         0.023945
min         0.400000
25%         0.443077
50%         0.454902
75%         0.466667
max         0.726667
Name: new_score, dtype: float64


In [13]:
thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.72, 0.74, 0.75, 0.80]

for t in thresholds:
    true_kept = (pairs_check["new_score"] >= t).sum()
    false_kept = (hard_negatives["new_score"] >= t).sum()

    true_recall = true_kept / len(pairs_check)
    false_rate = false_kept / len(hard_negatives)

    print(
        f"Threshold {t:.2f} | "
        f"True kept: {true_kept:4d} ({true_recall:.2%}) | "
        f"Wrong kept: {false_kept:4d} ({false_rate:.2%})"
    )

Threshold 0.50 | True kept: 3197 (89.08%) | Wrong kept:  503 (5.07%)
Threshold 0.55 | True kept: 3026 (84.31%) | Wrong kept:   50 (0.50%)
Threshold 0.60 | True kept: 2798 (77.96%) | Wrong kept:   10 (0.10%)
Threshold 0.65 | True kept: 2532 (70.55%) | Wrong kept:    4 (0.04%)
Threshold 0.70 | True kept: 2236 (62.30%) | Wrong kept:    1 (0.01%)
Threshold 0.72 | True kept: 2105 (58.65%) | Wrong kept:    1 (0.01%)
Threshold 0.74 | True kept: 1943 (54.14%) | Wrong kept:    0 (0.00%)
Threshold 0.75 | True kept: 1874 (52.22%) | Wrong kept:    0 (0.00%)
Threshold 0.80 | True kept: 1502 (41.85%) | Wrong kept:    0 (0.00%)


In [14]:
print(hard_negatives.columns.tolist())
print(hard_negatives.head(2))

['source1_entity_id', 'business_name', 's1_address', 'name_norm', 'matched_id', 'candidate_name', 'country', 'source', 'business_address', 'address_similarity', 'name_similarity', 'combined_score', 's1_addr_tokens', 'candidate_addr_tokens', 'address_token_jaccard', 's1_addr_numbers', 'candidate_addr_numbers', 'number_overlap', 'new_score']
  source1_entity_id                      business_name  \
0      S1-874660306  Physical Therapy Associates Group   
1      S1-874660306  Physical Therapy Associates Group   

                                  s1_address  \
0  4303 Elkins Avenue, Unit B, Nashville, TN   
1  4303 Elkins Avenue, Unit B, Nashville, TN   

                           name_norm    matched_id  \
0  physical therapy associates group  S2-289088001   
1  physical therapy associates group  S2-543386084   

                        candidate_name country source  \
0   Physical Therapy Associates  Group      US     S2   
1  Physical Therapy Associates (Group)      US     S2   

   

In [15]:
import numpy as np

thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.72, 0.74, 0.75]

results = []

# Ground truth lookup
gt_lookup = dict(
    zip(
        sample_gt["source1_entity_id"],
        sample_gt["matched_entity_ids"]
    )
)

# S1 IDs present in our validation
validation_s1_ids = set(sample_gt["source1_entity_id"])

for threshold in thresholds:

    f05_scores = []

    for s1_id in validation_s1_ids:

        # True IDs
        true_ids = [
            x.strip()
            for x in str(gt_lookup[s1_id]).split(",")
            if x.strip()
        ]

        # Candidates from true-pair table
        true_group = pairs_check[
            pairs_check["source1_entity_id"] == s1_id
        ]

        # Candidates from hard negatives
        false_group = hard_negatives[
            hard_negatives["source1_entity_id"] == s1_id
        ]

        all_candidates = pd.concat(
            [
                true_group[["matched_id", "new_score"]],
                false_group[["matched_id", "new_score"]]
            ],
            ignore_index=True
        )

        # Predict IDs above threshold
        pred_ids = all_candidates.loc[
            all_candidates["new_score"] >= threshold,
            "matched_id"
        ].drop_duplicates().tolist()

        f05 = calculate_f05(true_ids, pred_ids)
        f05_scores.append(f05)

    results.append({
        "threshold": threshold,
        "F0.5": np.mean(f05_scores)
    })

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

 threshold     F0.5
      0.50 0.896324
      0.55 0.916946
      0.60 0.885345
      0.65 0.838481
      0.70 0.778702
      0.72 0.749764
      0.74 0.714978
      0.75 0.698582


In [17]:
test_path = os.path.join(base, "test")

test_s1 = pd.read_csv(
    os.path.join(test_path, "test_source1.tsv"),
    sep="\t"
)

test_s2 = pd.read_csv(
    os.path.join(test_path, "test_source2.tsv"),
    sep="\t"
)

test_s3 = pd.read_csv(
    os.path.join(test_path, "test_source3.tsv"),
    sep="\t"
)

print("Test S1:", test_s1.shape)
print("Test S2:", test_s2.shape)
print("Test S3:", test_s3.shape)

print("\nColumns:")
print(test_s1.columns.tolist())
print(test_s2.columns.tolist())
print(test_s3.columns.tolist())

Test S1: (1732544, 4)
Test S2: (4887273, 4)
Test S3: (5082316, 4)

Columns:
['entity_id', 'business_name', 'business_address', 'country']
['entity_id', 'business_name', 'business_address', 'country']
['entity_id', 'business_name', 'business_address', 'country']


In [18]:
print("candidates:", candidates.shape)
print("ngram_to_ids:", len(ngram_to_ids))
print("block_name_to_ids:", len(block_name_to_ids))
print("sample_gt:", sample_gt.shape)
print("pairs_check:", pairs_check.shape)

candidates: (10320219, 8)
ngram_to_ids: 43338
block_name_to_ids: 7441525
sample_gt: (1000, 7)
pairs_check: (3589, 23)


In [20]:
# Test S2/S3 ke normalized names
test_s2["name_norm"] = test_s2["business_name"].map(normalize_name)
test_s3["name_norm"] = test_s3["business_name"].map(normalize_name)

# Candidate table
test_candidates = pd.concat([
    test_s2[["entity_id", "business_name", "business_address", "country", "name_norm"]],
    test_s3[["entity_id", "business_name", "business_address", "country", "name_norm"]]
], ignore_index=True)

# Country + first 3 characters block
test_candidates["ngram_block"] = (
    test_candidates["country"].fillna("").astype(str)
    + "_"
    + test_candidates["name_norm"].fillna("").str[:3]
)

# Block -> candidate IDs
test_ngram_to_ids = (
    test_candidates
    .groupby("ngram_block")["entity_id"]
    .apply(list)
    .to_dict()
)

print("Test candidates:", test_candidates.shape)
print("Test blocks:", len(test_ngram_to_ids))

Test candidates: (9969589, 6)
Test blocks: 55745


In [22]:
# Test S1 normalization + blocking

test_s1["name_norm"] = test_s1["business_name"].map(normalize_name)

test_s1["ngram_block"] = (
    test_s1["country"].fillna("").astype(str)
    + "_"
    + test_s1["name_norm"].fillna("").str[:3]
)

candidate_counts = (
    test_s1["ngram_block"]
    .map(lambda x: len(test_ngram_to_ids.get(x, [])))
)

print(candidate_counts.describe())

print("\nTotal candidate pairs:", candidate_counts.sum())
print("Average candidates per S1:", candidate_counts.mean())
print("Maximum candidates for one S1:", candidate_counts.max())

count    1.732544e+06
mean     9.850225e+03
std      1.269437e+04
min      0.000000e+00
25%      1.150000e+03
50%      7.283000e+03
75%      1.331400e+04
max      1.171060e+05
Name: ngram_block, dtype: float64

Total candidate pairs: 17065947542
Average candidates per S1: 9850.2246072827
Maximum candidates for one S1: 117106


In [24]:
from rapidfuzz import process, fuzz
import pandas as pd
import time

# 1. Block-wise candidates ready karo (sirf ek baar)
block_groups = {
    block: df
    for block, df in test_candidates.groupby("ngram_block", sort=False)
}

print("Blocks ready:", len(block_groups))

# 2. 10,000 S1 benchmark
test_sample = test_s1.head(10000)

total_before = 0
total_after = 0

start = time.time()

for row in test_sample.itertuples(index=False):

    block_df = block_groups.get(row.ngram_block)

    if block_df is None:
        continue

    total_before += len(block_df)

    # name -> entity_id mapping
    choices = block_df["name_norm"].tolist()

    results = process.extract(
        row.name_norm,
        choices,
        scorer=fuzz.ratio,
        limit=50,
        score_cutoff=50
    )

    total_after += len(results)

print("\nDONE")
print("Before:", total_before)
print("After:", total_after)
print("Average before:", total_before / len(test_sample))
print("Average after:", total_after / len(test_sample))
print("Time:", round(time.time() - start, 2), "seconds")

Blocks ready: 55745

DONE
Before: 98813044
After: 478216
Average before: 9881.3044
Average after: 47.8216
Time: 39.91 seconds


In [26]:
import gc

# Previous huge objects hatao
candidate_rows = []
match_rows = []

gc.collect()

print("Cleanup done")

Cleanup done


In [29]:
from rapidfuzz import process, fuzz
import pandas as pd
import re
import os
import time
import gc

TOP_K = 50
NAME_CUTOFF = 70
FINAL_THRESHOLD = 0.55

OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

candidate_path = f"{OUTPUT_DIR}/candidate_pairs.tsv"
matching_path = f"{OUTPUT_DIR}/matching_results.tsv"

# Old files hatao
for path in [candidate_path, matching_path]:
    if os.path.exists(path):
        os.remove(path)

# IMPORTANT:
# block_groups already memory mein hai.
# Agar nahi hai to ye ban jayega.
if "block_groups" not in globals():
    block_groups = {
        block: df
        for block, df in test_candidates.groupby("ngram_block", sort=False)
    }

print("Blocks:", len(block_groups))

# Output files
candidate_file = open(candidate_path, "w", encoding="utf-8")
matching_file = open(matching_path, "w", encoding="utf-8")

candidate_file.write(
    "source1_entity_id\tmatched_entity_id\n"
)

matching_file.write(
    "source1_entity_id\tmatched_entity_ids\n"
)

start = time.time()

for i, row in enumerate(test_s1.itertuples(index=False)):

    block_df = block_groups.get(row.ngram_block)

    if block_df is None:
        matching_file.write(f"{row.entity_id}\t\n")
        continue

    choices = block_df["name_norm"].tolist()

    # Top 50 by name
    results = process.extract(
        row.name_norm,
        choices,
        scorer=fuzz.ratio,
        limit=TOP_K,
        score_cutoff=NAME_CUTOFF
    )

    # Address calculation ke liye S1 features
    s1_addr_norm = normalize_text(row.business_address)
    s1_tokens = address_tokens(row.business_address)
    s1_numbers = address_numbers(row.business_address)

    selected = []

    for _, name_score, idx in results:

        cand = block_df.iloc[idx]

        name_sim = name_score / 100.0

        # Actual candidate fed to final scoring
        candidate_file.write(
            f"{row.entity_id}\t{cand.entity_id}\n"
        )

        # Address similarity
        cand_addr_norm = normalize_text(
            cand.business_address
        )

        addr_sim = fuzz.ratio(
            s1_addr_norm,
            cand_addr_norm
        ) / 100.0

        cand_tokens = address_tokens(
            cand.business_address
        )

        cand_numbers = address_numbers(
            cand.business_address
        )

        addr_jaccard = token_jaccard(
            s1_tokens,
            cand_tokens
        )

        num_overlap = number_overlap(
            s1_numbers,
            cand_numbers
        )

        score = (
            0.40 * name_sim
            + 0.20 * addr_sim
            + 0.20 * addr_jaccard
            + 0.20 * num_overlap
        )

        if score >= FINAL_THRESHOLD:
            selected.append(cand.entity_id)

    # Duplicate IDs remove
    selected = list(dict.fromkeys(selected))

    matching_file.write(
        f"{row.entity_id}\t{','.join(selected)}\n"
    )

    # Progress
    if (i + 1) % 10000 == 0:

        candidate_file.flush()
        matching_file.flush()

        elapsed = time.time() - start
        speed = (i + 1) / elapsed
        remaining = len(test_s1) - i - 1
        eta = remaining / speed

        print(
            f"{i+1:,}/{len(test_s1):,} | "
            f"{speed:.1f} S1/sec | "
            f"ETA: {eta/60:.1f} min"
        )

candidate_file.close()
matching_file.close()

gc.collect()

print("\n==============================")
print("SUBMISSION FILES CREATED")
print("==============================")
print(candidate_path)
print(matching_path)
print("Time:", round((time.time()-start)/60, 2), "minutes")

Blocks: 55745
10,000/1,732,544 | 131.7 S1/sec | ETA: 217.9 min
20,000/1,732,544 | 131.0 S1/sec | ETA: 217.9 min
30,000/1,732,544 | 131.4 S1/sec | ETA: 216.0 min


KeyboardInterrupt: 

In [30]:
import os

for f in [
    "/kaggle/working/output/matching_results.tsv",
    "/kaggle/working/output/candidate_pairs.tsv"
]:
    print("\n", f)
    print("Size:", round(os.path.getsize(f) / 1024 / 1024, 2), "MB")
    with open(f, "r", encoding="utf-8") as file:
        lines = sum(1 for _ in file)
    print("Lines:", lines)


 /kaggle/working/output/matching_results.tsv
Size: 2.15 MB
Lines: 39540

 /kaggle/working/output/candidate_pairs.tsv
Size: 34.77 MB
Lines: 1414483
